In [15]:
import os
import pickle
import logging
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from train_probe import ProbeTrainer
logger = logging.getLogger(__name__)

data_name = "int_sci_compare"
model_name = "Mistral-7B-v0.1"
num_layers = 32
data_dir = f"../data/{data_name}"
embed_dir = f"../embeddings/{model_name}/{data_name}"
output_dir = "../results"


In [2]:
trainer = ProbeTrainer(
    model_name=model_name,
    num_layers=num_layers,
    data_dir=data_dir,
    embed_dir=embed_dir,
    eval_test=False
)


2026-02-13 00:05:06,503 - INFO - Loaded 8000 train samples


2026-02-13 00:05:39,112 - INFO - Loaded train offset_0 embeddings: shape (32, 16000, 4096)
2026-02-13 00:06:11,797 - INFO - Loaded train offset_1 embeddings: shape (32, 16000, 4096)
2026-02-13 00:06:29,051 - INFO - Loaded train last_token embeddings: shape (32, 8000, 4096)
2026-02-13 00:06:29,110 - INFO - Loaded 1600 val samples
2026-02-13 00:06:36,931 - INFO - Loaded val offset_0 embeddings: shape (32, 3200, 4096)
2026-02-13 00:06:44,784 - INFO - Loaded val offset_1 embeddings: shape (32, 3200, 4096)
2026-02-13 00:06:49,342 - INFO - Loaded val last_token embeddings: shape (32, 1600, 4096)
2026-02-13 00:06:49,390 - INFO - Loaded 1600 test samples
2026-02-13 00:06:57,155 - INFO - Loaded test offset_0 embeddings: shape (32, 3200, 4096)
2026-02-13 00:07:04,939 - INFO - Loaded test offset_1 embeddings: shape (32, 3200, 4096)
2026-02-13 00:07:09,547 - INFO - Loaded test last_token embeddings: shape (32, 1600, 4096)
2026-02-13 00:07:09,556 - INFO - Dataset: int_sci_compare
2026-02-13 00:07:0

In [6]:
load_dir = os.path.join(output_dir, model_name, trainer.dataset_name)
reg_probes = trainer.load_probes(load_dir, 'regression')
cls_probes = trainer.load_probes(load_dir, 'classification')



2026-02-13 00:12:00,086 - INFO - Loaded regression probes from ../results/Mistral-7B-v0.1/int_sci_compare/regression_probes.pkl
2026-02-13 00:12:00,125 - INFO - Loaded classification probes from ../results/Mistral-7B-v0.1/int_sci_compare/classification_probes.pkl


In [7]:
val_reg_data = trainer.prepare_regression_data('val')
test_reg_data = trainer.prepare_regression_data('test')

In [8]:
val_reg_data.keys()

dict_keys(['int_offset_0', 'sci_offset_0', 'mixed_offset_0', 'int_offset_1', 'sci_offset_1', 'mixed_offset_1'])

In [9]:
reg_probes.keys()

dict_keys(['int_offset_0', 'sci_offset_0', 'mixed_offset_0', 'int_offset_1', 'sci_offset_1', 'mixed_offset_1'])

In [10]:
# RE stands for relative error, AE stands for absolute error
import numpy as np
results = {}

for probe_name, models in reg_probes.items():
    if probe_name not in val_reg_data:
        continue

    X_val = val_reg_data[probe_name]['X']
    y_val = val_reg_data[probe_name]['y']

    X_test = test_reg_data[probe_name]['X']
    y_test = test_reg_data[probe_name]['y']

    probe_results = []
    for layer_idx, model in enumerate(models):
        y_val_pred = model.predict(X_val[layer_idx])
        y_test_pred = model.predict(X_test[layer_idx])
        probe_results.append({
            'val_median_RE_real': np.median(np.abs(np.exp2(y_val) - np.exp2(y_val_pred)) / np.abs(np.exp2(y_val))),
            'val_RE_median_symmetric': np.exp2(np.median(np.abs(y_val_pred-y_val))) - 1,
            'val_median_RE_symmetric': np.median(np.exp2((np.abs(y_val_pred-y_val))) - 1),
            'test_median_RE_real': np.median(np.abs(np.exp2(y_test) - np.exp2(y_test_pred)) / np.abs(np.exp2(y_test))),
            'test_RE_median_symmetric': np.exp2(np.median(np.abs(y_test_pred-y_test))) - 1,
            'test_median_RE_symmetric': np.median(np.exp2((np.abs(y_test_pred-y_test))) - 1)
        })

    results[probe_name] = probe_results

In [14]:
VAL_METRICS = [
    "val_median_RE_real",
    "val_RE_median_symmetric",
    "val_median_RE_symmetric",
]

TEST_METRICS = [
    "test_median_RE_real",
    "test_RE_median_symmetric",
    "test_median_RE_symmetric",
]

# map val metric -> corresponding test metric
VAL_TO_TEST = {
    "val_median_RE_real": "test_median_RE_real",
    "val_RE_median_symmetric": "test_RE_median_symmetric",
    "val_median_RE_symmetric": "test_median_RE_symmetric",
}

print("Pick best layer by VAL metric; report both VAL and TEST at that layer (lower is better)\n")

for probe_type, layer_list in results.items():
    if not layer_list:
        continue

    print(f"--- probe type: {probe_type}")

    for val_metric in VAL_METRICS:
        test_metric = VAL_TO_TEST[val_metric]

        val_vals = np.array([d[val_metric] for d in layer_list], dtype=float)
        best_layer = int(np.argmin(val_vals))

        best_val = layer_list[best_layer][val_metric]
        best_test = layer_list[best_layer][test_metric]

        print(
            f"  select by {val_metric}: "
            f"layer={best_layer}, val={best_val:.6g}, test={best_test:.6g}"
        )


Pick best layer by VAL metric; report both VAL and TEST at that layer (lower is better)

--- probe type: int_offset_0
  select by val_median_RE_real: layer=24, val=0.0550975, test=0.0581755
  select by val_RE_median_symmetric: layer=24, val=0.0561104, test=0.0594493
  select by val_median_RE_symmetric: layer=24, val=0.0561104, test=0.0594493
--- probe type: sci_offset_0
  select by val_median_RE_real: layer=31, val=0.0226384, test=0.0219685
  select by val_RE_median_symmetric: layer=31, val=0.022981, test=0.022107
  select by val_median_RE_symmetric: layer=31, val=0.022981, test=0.022107
--- probe type: mixed_offset_0
  select by val_median_RE_real: layer=27, val=0.0568144, test=0.0553076
  select by val_RE_median_symmetric: layer=27, val=0.0584334, test=0.0571244
  select by val_median_RE_symmetric: layer=27, val=0.0584334, test=0.0571244
--- probe type: int_offset_1
  select by val_median_RE_real: layer=31, val=0.0496909, test=0.0516458
  select by val_RE_median_symmetric: layer=31, 